In [1]:
import cv2
import numpy as np
import datetime
import os

In [2]:
def preprocess_image(frame):
    """
    Aplica filtros de suavização e binarização adaptativa.
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # Filtro Gaussiano para reduzir ruído da webcam
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    # Threshold Adaptativo para lidar com sombras na sala de aula
    thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                   cv2.THRESH_BINARY_INV, 11, 2)
    return thresh

In [3]:
def get_contours(thresh):
    """
    Extrai contornos para localizar a folha ou as questões.
    """
    # Operação Morfológica: Dilatação para fechar pequenos buracos nas marcações
    kernel = np.ones((3,3), np.uint8)
    dilated = cv2.dilate(thresh, kernel, iterations=1)
    
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return contours

In [4]:
def main():
    # Inicializa a captura da Webcam
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Erro: Não foi possível abrir a webcam.")
        return

    print("Pressione 'q' para sair ou 's' para salvar um snapshot.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # 1. Pré-processamento (Filtros e Cores)
        processed = preprocess_image(frame)

        # 2. Detecção de Contornos (Localização)
        contours = get_contours(processed)

        # Visualização básica: Desenha contornos maiores (possivelmente a folha)
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area > 5000: # Filtra ruídos pequenos
                cv2.drawContours(frame, [cnt], -1, (0, 255, 0), 3)
                
                # Aqui entrará a lógica de extração de ROIs e Watershed nas próximas fases
                x, y, w, h = cv2.boundingRect(cnt)
                cv2.putText(frame, "Folha Detectada", (x, y-10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        # Exibição dos resultados (On-the-fly)
        cv2.imshow("SPV - Corretor de Gabaritos (Original)", frame)
        cv2.imshow("SPV - Visao Binaria (Processada)", processed)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('s'):
            timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
            cv2.imwrite(f"../data/images/snapshot_gabarito_{timestamp}.png", frame)
            print("Imagem salva!")

    cap.release()
    cv2.destroyAllWindows()

In [5]:
if __name__ == "__main__":
    main()

Pressione 'q' para sair ou 's' para salvar um snapshot.


QFontDatabase: Cannot find font directory /home/ufabc/miniconda3/envs/PDI26/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/ufabc/miniconda3/envs/PDI26/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/ufabc/miniconda3/envs/PDI26/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/ufabc/miniconda3/envs/PDI26/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find f